# Analysez les données de systèmes éducatifs

In [25]:
import pandas as pd

imported_countries = pd.read_csv('data/raw/EdStatsCountry.csv')
imported_country_series = pd.read_csv('data/raw/EdStatsCountry-Series.csv')
imported_data = pd.read_csv('data/raw/EdStatsData.csv')
imported_footnote = pd.read_csv('data/raw/EdStatsFootNote.csv')
imported_series = pd.read_csv('data/raw/EdStatsSeries.csv')

### Réalisez votre premier nettoyage

In [26]:
# Supression des colonnes inconnues
imported_series.drop('Unnamed: 20', axis=1, inplace=True)
imported_footnote.drop('Unnamed: 4', axis=1, inplace=True)
imported_data.drop('Unnamed: 69', axis=1, inplace=True)
imported_country_series.drop('Unnamed: 3', axis=1, inplace=True)
imported_countries.drop('Unnamed: 31', axis=1, inplace=True)

# Renommage des colonnes de releration
imported_series.rename(columns={'Series Code' : 'SeriesCode'}, inplace=True)
imported_data.rename(columns={'Indicator Code' : 'SeriesCode', 'Country Code': 'CountryCode'}, inplace=True)
imported_countries.rename(columns={'Country Code' : 'CountryCode'}, inplace=True)

#### Supression des faux pays

J'ai remarqué ques les faux pays sont ceux du jeu de donnée qui n'ont pas de région associée. On peut filtrer les pays de cette manière.

In [27]:
print(f'Countries avant : {imported_countries.shape}')
countries_cleaned = imported_countries[ ~ imported_countries['Region'].isnull()]
print(f'Countries après : {countries_cleaned.shape}')

Countries avant : (241, 31)
Countries après : (214, 31)


#### Supression des faux pays dans les autres df

##### Méthode 1

En stockant les faux pays dans une liste d'exclusion qui sera utilisée pour le filtrage dans les autres fichiers.

In [28]:
# On enregistre les pays a supprimer
print(f'Nombre de pays total : {imported_countries.shape[0]}')
countries_to_remove = imported_countries[imported_countries['Region'].isnull()]
print(f'Country a supprimer : {countries_to_remove.shape[0]}')
print(f'Country restant : {imported_countries.shape[0] - countries_to_remove.shape[0]}')

# On supprime les country series dont le countryCode est dans la liste des pays à supprimer
print(f'Country series avant : {imported_country_series.shape}')
method_one_country_series_filtered = imported_country_series[ ~ imported_country_series['CountryCode'].isin(countries_to_remove['CountryCode'])]
print(f'Country series après : {method_one_country_series_filtered.shape}')

# On supprime les stats data dont le countryCode est dans la liste des pays à supprimer
print(f'Stats data avant : {imported_data.shape}')
method_one_stats_filtered = imported_data[ ~ imported_data['CountryCode'].isin(countries_to_remove['CountryCode'])]
print(f'Stats data après : {method_one_stats_filtered.shape}')

# On supprime les footnote dont le countryCode est dans la liste des pays à supprimer
print(f'Footnote avant : {imported_footnote.shape}')
method_one_footnote_filtered = imported_footnote[ ~ imported_footnote['CountryCode'].isin(countries_to_remove['CountryCode'])]
print(f'Footnote après : {method_one_footnote_filtered.shape}')

Nombre de pays total : 241
Country a supprimer : 27
Country restant : 214
Country series avant : (613, 3)
Country series après : (611, 3)
Stats data avant : (886930, 69)
Stats data après : (787975, 69)
Footnote avant : (643638, 4)
Footnote après : (516743, 4)


##### Méthode 2

En utilisant des fusions entre les différents fichiers à partir des pays déja filtré Country.

In [29]:
# On fusionne le df country au df country_series sur la clé CountryCode
print(f'Country series avant : {imported_country_series.shape}')
method_two_country_series_fitered = imported_country_series.merge(countries_cleaned, how='inner')
print(f'Country series avant : {method_two_country_series_fitered.shape}')

# On fusionne le df imported_data au df countries_cleaned sur la clé CountryName
print(f'Stats data avant : {imported_data.shape}')
method_two_stats_filtered = imported_data.merge(countries_cleaned, how='inner')
print(f'Stats data après : {method_one_stats_filtered.shape}')

# On fusionne le df imported_footnote au df countries_cleaned sur la clé CountryName
print(f'Footnote avant : {imported_footnote.shape}')
method_two_footnote_filtered = imported_footnote.merge(countries_cleaned, how='inner')
print(f'Footnote après : {method_one_footnote_filtered.shape}')

Country series avant : (613, 3)
Country series avant : (611, 33)
Stats data avant : (886930, 69)
Stats data après : (787975, 69)
Footnote avant : (643638, 4)
Footnote après : (516743, 4)


### Sélection des indicateurs

In [30]:
# Parmi les jeux de données centrés sur les indicateurs, identifiez la colonne qui décrit la catégorie métier à laquelle appartient chaque indicateur.
imported_series['Topic'].value_counts()
print(f'Liste des catégories métier des indicateurs : {imported_series['Topic'].unique().tolist()}')

Liste des catégories métier des indicateurs : ['Attainment', 'Education Equality', 'Infrastructure: Communications', 'Learning Outcomes', 'Economic Policy & Debt: National accounts: US$ at current prices: Aggregate indicators', 'Economic Policy & Debt: National accounts: US$ at constant 2010 prices: Aggregate indicators', 'Economic Policy & Debt: Purchasing power parity', 'Economic Policy & Debt: National accounts: Atlas GNI & GNI per capita', 'Teachers', 'Education Management Information Systems (SABER)', 'Early Child Development (SABER)', 'Engaging the Private Sector (SABER)', 'School Health and School Feeding (SABER)', 'School Autonomy and Accountability (SABER)', 'School Finance (SABER)', 'Student Assessment (SABER)', 'Teachers (SABER)', 'Tertiary Education (SABER)', 'Workforce Development (SABER)', 'Literacy', 'Background', 'Primary', 'Secondary', 'Tertiary', 'Early Childhood Education', 'Pre-Primary', 'Expenditures', 'Health: Risk factors', 'Health: Mortality', 'Social Protection

Catégories utiles :
- Learning Outcomes (Résultats d'apprentissage)
- Attainment (Niveau d'instruction (ou Niveau d'études atteint)
- Secondary (Secondaire)
- Tertiary (Supérieur)
- Teachers (Enseignants)
- Literacy (Alphabétisation)
- Infrastructure: Communications
- Population (population)

In [31]:
# Gardez les catégories qui font sens par rapport à la demande de Mark et l’objectif du projet et supprimez les autres.
significant_topics = ['Learning Outcomes', 'Attainment', 'Secondary', 'Tertiary', 'Teachers', 'Literacy', 'Infrastructure', 'Population']
significative_series = imported_series[imported_series['Topic'].isin(significant_topics)]

# Calculez le nombre d’indicateurs restants.
print(f'Indicateurs avant filtrage : {imported_series.shape}')
print(f'Indicateurs après filtrage : {significative_series.shape}')

print('Nombre d\'indicateur par catégorie : ')
significative_series['Topic'].value_counts()


Indicateurs avant filtrage : (3665, 20)
Indicateurs après filtrage : (2575, 20)
Nombre d'indicateur par catégorie : 


Topic
Learning Outcomes    1046
Attainment            733
Secondary             256
Population            213
Tertiary              158
Teachers              137
Literacy               32
Name: count, dtype: int64

In [32]:
# Filtrez l’ensemble des jeux de données pour ne garder que les indicateurs sélectionnés.
print(f'Country series avant : {method_one_country_series_filtered.shape}')
country_series_filtered = method_one_country_series_filtered[method_one_country_series_filtered['SeriesCode'].isin(significative_series['SeriesCode'])]
print(f'Country series après : {country_series_filtered.shape}')

print(f'Stats data avant : {method_one_stats_filtered.shape}')
stats_filtered = method_one_stats_filtered[method_one_stats_filtered['SeriesCode'].isin(significative_series['SeriesCode'])]
print(f'Stats data après : {stats_filtered.shape}')

print(f'Footnote avant : {method_one_footnote_filtered.shape}')
footnote_filtered = method_one_footnote_filtered[method_one_footnote_filtered['SeriesCode'].isin(significative_series['SeriesCode'])]
print(f'Footnote après : {footnote_filtered.shape}')

Country series avant : (611, 3)
Country series après : (0, 3)
Stats data avant : (787975, 69)
Stats data après : (547175, 69)
Footnote avant : (516743, 4)
Footnote après : (278882, 4)


## Supression des années

Dans le fichier Data, interprétez les colonnes représentant les années.

Pourquoi avons-nous des valeurs d’indicateur pour des années futures ?
A but de projections futures

In [33]:
# les indicateurs ayant des donnée renseigné pour l'année 2100 par exemple "Wittgenstein Projection: Mean years of schooli..."
imported_data.loc[~imported_data['2100'].isnull()]

,Country Name,CountryCode,Indicator Name,SeriesCode,1970,1971,1972,1973,1974,1975,...,2055,2060,2065,2070,2075,2080,2085,2090,2095,2100
91309,World,WLD,Wittgenstein Projection: Mean years of schooli...,PRJ.MYS.0T19.FE,NaN,NaN,NaN,NaN,NaN,NaN,...,2.50,2.50,2.50,2.60,2.60,2.60,2.70,2.70,2.70,2.70
91310,World,WLD,Wittgenstein Projection: Mean years of schooli...,PRJ.MYS.0T19.MA,NaN,NaN,NaN,NaN,NaN,NaN,...,2.40,2.50,2.50,2.60,2.60,2.60,2.70,2.70,2.70,2.70
91311,World,WLD,Wittgenstein Projection: Mean years of schooli...,PRJ.MYS.0T19.MF,NaN,NaN,NaN,NaN,NaN,NaN,...,2.40,2.50,2.50,2.60,2.60,2.60,2.70,2.70,2.70,2.70
91312,World,WLD,Wittgenstein Projection: Mean years of schooli...,PRJ.MYS.15UP.FE,NaN,NaN,NaN,NaN,NaN,NaN,...,10.60,10.80,11.10,11.40,11.60,11.90,12.10,12.30,12.50,12.70
91313,World,WLD,Wittgenstein Projection: Mean Years of Schooli...,PRJ.MYS.15UP.GPI,NaN,NaN,NaN,NaN,NaN,NaN,...,0.24,0.21,0.17,0.15,0.12,0.11,0.10,0.09,0.08,0.08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886917,Zimbabwe,ZWE,Wittgenstein Projection: Population in thousan...,PRJ.POP.ALL.1.MA,NaN,NaN,NaN,NaN,NaN,NaN,...,253.28,212.14,179.40,150.11,126.43,106.82,91.70,78.21,67.56,59.06
886918,Zimbabwe,ZWE,Wittgenstein Projection: Population in thousan...,PRJ.POP.ALL.1.MF,NaN,NaN,NaN,NaN,NaN,NaN,...,574.00,494.23,420.47,352.09,294.10,244.68,205.47,171.39,144.86,124.17
886919,Zimbabwe,ZWE,Wittgenstein Projection: Population in thousan...,PRJ.POP.ALL.3.FE,NaN,NaN,NaN,NaN,NaN,NaN,...,3434.09,3469.66,3493.32,3491.72,3468.64,3429.54,3378.61,3317.67,3248.91,3175.64
886920,Zimbabwe,ZWE,Wittgenstein Projection: Population in thousan...,PRJ.POP.ALL.3.MA,NaN,NaN,NaN,NaN,NaN,NaN,...,3124.32,3106.84,3078.78,3041.50,3000.74,2961.23,2924.73,2889.02,2851.65,2813.31


In [34]:
# Sur la base de votre compréhension de la problématique métier, filtrez les années en conséquence.
# Je supprime les colonnes des années au delas de 2020 et avant 2010

import numpy as np

# On retire les années après 2016 qui sont des projections
columns_to_drop = np.concatenate((np.arange(1970,2010,1), np.arange(2020,2105,5)))

columns_to_drop = [str(col) for col in columns_to_drop]
print(f'Shape avant : {stats_filtered.shape}')
stats_filtered.drop(columns=columns_to_drop, inplace=True)
print(f'Shape après : {stats_filtered.shape}')

Shape avant : (547175, 69)
Shape après : (547175, 12)


## Supression des indicateurs à faible taux de remplissage

In [35]:
years = np.arange(2010,2018, 1)
years_col = [str(col) for col in years]

In [36]:
# Pour chaque année, calculez la proportion d’indicateurs avec des valeurs renseignées (c’est-à-dire, non manquantes).
stats_by_year = stats_filtered[years_col]
stats_by_year.notnull().mean()

# Nous n'avons aucune donnée pour l'année 2017

2010    0.338356
2011    0.162316
2012    0.166400
2013    0.152872
2014    0.126285
2015    0.204437
2016    0.012075
2017    0.000000
dtype: float64

In [37]:
# On remarque que l'année 2017 est totalement vide, on la supprime
stats_filtered.drop(columns=['2017'], inplace=True)
years_col.remove('2017')

In [38]:
# Pour chaque indicateur, calculez la proportion d’années avec des valeurs renseignées.
stats_by_year = stats_filtered[years_col]

stats_filtered['percentage_data_filled'] = stats_by_year.notnull().mean(axis=1)
stats_filtered

,Country Name,CountryCode,Indicator Name,SeriesCode,2010,2011,2012,2013,2014,2015,2016,percentage_data_filled
91625,Afghanistan,AFG,"Adjusted net enrolment rate, lower secondary, ...",UIS.NERA.2,NaN,NaN,NaN,47.436790,50.627232,NaN,NaN,0.285714
91626,Afghanistan,AFG,"Adjusted net enrolment rate, lower secondary, ...",UIS.NERA.2.F,NaN,NaN,NaN,34.073261,37.641541,NaN,NaN,0.285714
91627,Afghanistan,AFG,"Adjusted net enrolment rate, lower secondary, ...",UIS.NERA.2.GPI,NaN,NaN,NaN,0.567060,0.598370,NaN,NaN,0.285714
91628,Afghanistan,AFG,"Adjusted net enrolment rate, lower secondary, ...",UIS.NERA.2.M,NaN,NaN,NaN,60.087059,62.906952,NaN,NaN,0.285714
91633,Afghanistan,AFG,"Adjusted net enrolment rate, upper secondary, ...",UIS.NERA.3,NaN,NaN,NaN,31.332621,32.417030,NaN,NaN,0.285714
...,...,...,...,...,...,...,...,...,...,...,...,...
886922,Zimbabwe,ZWE,"Youth illiterate population, 15-24 years, % fe...",UIS.LPP.AG15T24,NaN,43.61436,NaN,NaN,35.887100,NaN,NaN,0.285714
886926,Zimbabwe,ZWE,"Youth literacy rate, population 15-24 years, b...",SE.ADT.1524.LT.ZS,NaN,90.93070,NaN,NaN,90.428120,NaN,NaN,0.285714
886927,Zimbabwe,ZWE,"Youth literacy rate, population 15-24 years, f...",SE.ADT.1524.LT.FE.ZS,NaN,92.12456,NaN,NaN,93.188350,NaN,NaN,0.285714
886928,Zimbabwe,ZWE,"Youth literacy rate, population 15-24 years, g...",SE.ADT.1524.LT.FM.ZS,NaN,1.02828,NaN,NaN,1.063890,NaN,NaN,0.285714


In [39]:
# Identifiez les indicateurs particulièrement riches en données en calculant un dataframe contenant par indicateur et par année, le nombre de pays avec une valeur renseignée.

indicator_with_data_for_country = stats_filtered.groupby(['Indicator Name'])[years_col].agg('count')
indicator_with_data_for_country

,2010,2011,2012,2013,2014,2015,2016
Indicator Name,,,,,,,
"Adjusted net enrolment rate, lower secondary, both sexes (%)",106,107,101,99,84,3,0
"Adjusted net enrolment rate, lower secondary, female (%)",103,106,100,98,83,3,0
"Adjusted net enrolment rate, lower secondary, gender parity index (GPI)",103,106,100,98,83,3,0
"Adjusted net enrolment rate, lower secondary, male (%)",103,106,100,98,83,3,0
"Adjusted net enrolment rate, upper secondary, both sexes (%)",57,57,52,100,89,3,0
...,...,...,...,...,...,...,...
"Youth illiterate population, 15-24 years, % female",47,57,41,29,35,31,17
"Youth literacy rate, population 15-24 years, both sexes (%)",47,57,41,29,36,32,17
"Youth literacy rate, population 15-24 years, female (%)",47,57,41,29,36,32,17


In [40]:
indicator_with_data_for_country['total_countries'] = indicator_with_data_for_country[years_col].mean(axis=1)

# On ne garde que les indicateurs qui ont en moyenne au moins 100 pays dont la valeur est renseignée
indicators_filtered = indicator_with_data_for_country.loc[indicator_with_data_for_country['total_countries'] > 100].sort_values('total_countries', ascending=False)
indicators_filtered

,2010,2011,2012,2013,2014,2015,2016,total_countries
Indicator Name,,,,,,,,
Theoretical duration of upper secondary education (years),203,202,202,203,203,203,173,198.428571
Theoretical duration of secondary education (years),203,202,202,203,203,203,173,198.428571
Official entrance age to lower secondary education (years),203,202,202,203,203,203,173,198.428571
"Population of the official entrance age to secondary general education, both sexes (number)",190,193,192,192,190,185,186,189.714286
"Population of the official age for upper secondary education, both sexes (number)",194,195,195,195,196,194,157,189.428571
...,...,...,...,...,...,...,...,...
"Enrolment in Grade 2 of lower secondary general education, female (number)",146,149,145,139,128,8,0,102.142857
"Enrolment in upper secondary education, female (number)",149,152,150,140,114,7,0,101.714286
"Gross enrolment ratio, lower secondary, gender parity index (GPI)",146,153,147,139,119,7,0,101.571429


Voici la liste des indicateurs sélectionnés qui me semblent pertinents :
- Population of the official age for upper secondary education, both sexes (Lycéens potentiels)
- Population of the official age for tertiary education, both sexes (Étudiants potentiels)
- Population, ages 15-24, total (La tranche d'âge globale cible pour le Lycée et supérieur)
- Enrolment in secondary education, both sexe (Nombre réel d'inscrits)
- Gross enrolment ratio, upper secondary, both sexes (Taux de scolarisation au lycée)
- Gross enrolment ratio, secondary, both sexes (%)


In [41]:
# Filtrez votre dataframe Data pour ne garder que les indicateurs, pays et années que vous avez jugé pertinents sur la base de vos analyses précédentes.

indicators_selected = [
    'Population of the official age for upper secondary education, both sexes (number)',
    'Population of the official age for tertiary education, both sexes (number)',
    'Enrolment in secondary education, both sexes (number)',
    'Gross enrolment ratio, upper secondary, both sexes (%)',
    'Gross enrolment ratio, secondary, both sexes (%)',
    'Population, age 15, total',
    'Population, age 16, total',
    'Population, age 17, total',
    'Population, age 18, total',
    'Population, age 19, total',
    'Population, age 20, total',
    'Population, age 21, total',
    'Population, age 22, total',
    'Population, age 23, total',
    'Population, age 24, total',
]

final_data = stats_filtered.loc[stats_filtered['Indicator Name'].isin(indicators_selected)]

In [42]:
final_data['nb_empty_cells'] = final_data[years_col].isnull().sum(axis=1)

empty_cells_by_country = final_data.groupby('Country Name').agg({'nb_empty_cells': 'sum'}).sort_values('nb_empty_cells', ascending=False)
empty_cells_by_country = empty_cells_by_country.reset_index()

print(empty_cells_by_country)

# On remarque que certains pays, on a beaucoup de donnée vide
# On a 15 indicateurs pour 7 années donc 105 données au max par pays
# On supprime tous les pays qui ont plus de 50% de valeurs vides, car ça risque de fausser les moyenne lors de l'agreggation.
countries_to_keep = empty_cells_by_country[empty_cells_by_country['nb_empty_cells'] < (0.5 * 105)]

print(f'Pays avant traitement : {final_data['Country Name'].unique().shape[0]}')
final_data_without_empty_countries = final_data.loc[final_data['Country Name'].isin(countries_to_keep['Country Name'])]
print(f'Pays avant traitement : {final_data_without_empty_countries['Country Name'].unique().shape[0]}')

              Country Name  nb_empty_cells
0         French Polynesia             105
1                   Kosovo             105
2          Channel Islands             105
3           American Samoa             105
4                     Guam             105
..                     ...             ...
210  Sao Tome and Principe              22
211                  Nepal              22
212                Ecuador              21
213             Kazakhstan              20
214             Uzbekistan              20

[215 rows x 2 columns]
Pays avant traitement : 215
Pays avant traitement : 180


In [43]:
# Suppression des colonnes qui ne sont plus utiles à ce stade
final_data_without_empty_countries.drop(columns=['CountryCode', 'SeriesCode'], inplace=True)

In [44]:
# Agrégez ce dataframe pour en construire un nouveau : chaque ligne doit correspondre à un pays et chaque colonne doit correspondre à un indicateur.
final_data_without_empty_countries['mean_2010_2016'] = final_data_without_empty_countries[years_col].mean(axis=1)
data_formated = final_data_without_empty_countries.pivot_table(index="Country Name", columns="Indicator Name", values='mean_2010_2016', aggfunc='mean')

data_formated.head()

Indicator Name,"Enrolment in secondary education, both sexes (number)","Gross enrolment ratio, secondary, both sexes (%)","Gross enrolment ratio, upper secondary, both sexes (%)","Population of the official age for tertiary education, both sexes (number)","Population of the official age for upper secondary education, both sexes (number)","Population, age 15, total","Population, age 16, total","Population, age 17, total","Population, age 18, total","Population, age 19, total","Population, age 20, total","Population, age 21, total","Population, age 22, total","Population, age 23, total","Population, age 24, total"
Country Name,,,,,,,,,,,,,,,
Afghanistan,2.418162e+06,55.421597,41.557182,2.819768e+06,2.035841e+06,772546.4,749435.4,725380.6,699724.6,673189.8,647178.0,621438.8,596597.6,573119.6,550735.6
Albania,3.425055e+05,93.337434,88.474256,2.751482e+05,1.652629e+05,54522.8,55721.4,56953.0,58207.6,59379.4,60430.2,61438.4,61727.4,60980.8,59507.2
Algeria,4.594370e+06,98.516056,61.089111,3.696664e+06,1.992388e+06,625772.2,640533.8,656372.8,672386.8,688118.4,703946.6,719714.6,731856.4,738597.6,741010.0
Angola,8.676580e+05,28.840014,20.749240,2.168874e+06,1.556589e+06,451662.8,440216.8,428840.2,417139.8,405217.6,393686.8,382576.4,370579.8,357158.2,342879.8
Argentina,4.339148e+06,104.738878,80.659679,3.444851e+06,2.069631e+06,686746.2,690930.4,693496.6,693510.6,691643.2,689868.2,688234.2,684743.4,678743.0,671282.8


In [45]:
import os

if not os.path.exists('./data/interim/'):
    os.mkdir('./data/interim/')

data_formated.to_csv('./data/interim/EdStatsData.csv')